# Which new study programme for the EMI faculty? Progress notebook

**Question.** Which new degree programme would bring the most *new* students to OTH Amberg-Weiden and can be taught by the EMI faculty with the staff and labs it already has?

**How we work.** The programme has to come out of the data. We do not start from programmes we picked ourselves. The work runs in nine steps, one at a time. Every step has rules that were written down before the step ran.

This notebook grows by one section per finished step, so it always shows where the project stands.

- Rules: `configs/funnel.yaml`
- Why each rule is there: `docs/funnel_preregistration.md`
- Decisions with dates: `docs/decision_log.md`

In [1]:
import os, sys, re, yaml, pandas as pd
from pathlib import Path
from IPython.display import display, Markdown

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT / "src"))
pd.set_option("display.max_colwidth", None); pd.set_option("display.width", 200)

def show(df):
    display(df.style.hide(axis="index").set_properties(**{"text-align": "left", "white-space": "normal"})
            .set_table_styles([{"selector": "th", "props": [("text-align", "left")]}]))

RULES_TEXT = (ROOT / "configs" / "funnel.yaml").read_text(encoding="utf-8")
RULES = yaml.safe_load(RULES_TEXT)
print("Rules loaded:", len(RULES), "blocks -", ", ".join(RULES))

Rules loaded: 17 blocks - definitions, universe, text, similarity, discovery, pool_mapping, screen, freeze, demand, labour, capture, shortlist, feasibility_gate, ranking, stress_tests, decision, conformance


## Where we are

In [2]:
STATUS = [   # updated after every finished step
    (0, "Rules", "Which rules decide?", "done"),
    (1, "Data inventory", "Is the data there and current?", "next"),
    (2, "Programme types", "What kinds of programme exist in Germany?", "open"),
    (3, "Screen", "Which types are open to OTH and in EMI's field? How much competition is near?", "open"),
    (4, "Student trend", "Are students coming?", "open"),
    (5, "Job ads", "Are there jobs nearby?", "open"),
    (6, "New students", "How many first-years would be new to OTH?", "open"),
    (7, "Feasibility", "Can EMI teach it with what it has?", "open"),
    (8, "Decision", "Which programme, and how sure are we?", "open"),
]
show(pd.DataFrame(STATUS, columns=["Step", "Name", "Question", "Status"]))

Step,Name,Question,Status
0,Rules,Which rules decide?,done
1,Data inventory,Is the data there and current?,next
2,Programme types,What kinds of programme exist in Germany?,open
3,Screen,Which types are open to OTH and in EMI's field? How much competition is near?,open
4,Student trend,Are students coming?,open
5,Job ads,Are there jobs nearby?,open
6,New students,How many first-years would be new to OTH?,open
7,Feasibility,Can EMI teach it with what it has?,open
8,Decision,"Which programme, and how sure are we?",open


## Why we started again

- Our first analysis started from programmes **we** had chosen (cybersecurity, robotics). That is a biased start: the result can only be one of our own ideas.
- A first data-driven run was reviewed and is **not** used as a result. The code had applied several rules differently from how they were written, and one test (competition) was not valid.
- So the analysis is redone step by step. The rules come first, and each step is checked before the next one starts.

## What the supervisor told us (meeting and reply email)

| The supervisor said | What we do with it |
|---|---|
| An entire new degree programme, not a few extra modules. It must not already exist at OTH. | Only whole programmes are candidates. Anything OTH already offers is removed. |
| Bachelor or master, both are fine. | Both levels are analysed. |
| An overlap of 40-80 % with existing lectures is ideal. Too much overlap is just a relabelled programme. Too little costs too many resources. | At least 40 % of the credits must be covered by modules EMI already teaches. At most 80 % may come from one existing programme. |
| He has no minimum number of students. We should assume students come and **argue why**. | We keep 20 first-years as our own assumption and show what changes with 15, 25 or no minimum. |
| It need not be unique in Germany. Programmes abroad are fine as inspiration. | Competition is measured and used as an argument. It does not remove a candidate by itself. |
| Good arguments: it does not exist in Germany, or a few similar programmes elsewhere (for example in the north) attract many students and there is nothing comparable in the south. | Every candidate gets a north/south check: does it exist elsewhere, is it missing in Bavaria and the south, and how well can we show it is popular there. |
| He cannot give student numbers per course. He gave us the lecture timetables of the faculty. | The timetables show which topics current lecturers already teach. They stay outside the repository. |
| Sources: other universities and the Federal Statistical Office. | Unchanged: Destatis, the national study catalogue, other universities' module handbooks, the job board of the employment agency. |
| What counts is the chain of reasoning and the documentation. | Every step ends with short notes on why each programme stays or goes. |

## Step 0: the rules, fixed before any data is touched

**Question.** Which rules decide, so that nobody (including us) can bend them later to get a favourite result?

**What we did.** We wrote every rule and every threshold into one file and committed it before running any analysis. The numbers below are read from that file.

In [3]:
g, c, s, d = RULES["feasibility_gate"], RULES["capture"], RULES["screen"], RULES["decision"]["rule"]
rows = [
    (2, "Programme types", "All full-time bachelor and master offers in Germany are grouped by their text. No topic list is written by hand.",
        f"a group is stable from an agreement of {RULES['discovery']['stable_jaccard']} across repeated runs"),
    (3, "Already at OTH", "A type is removed if an OTH programme of that level sits in its centre.", "closer than half of the type's own programmes"),
    (3, "Teacher training", "Removed: a faculty cannot open a state teaching degree on its own.", f"from a share of {s['teacher_training']['drop_if_at_least']:.0%}"),
    (3, "Shrinking field", "Removed if German first-years in Bavaria fall and the trend has predicted well in the past.", f"below {s['trend']['drop_if_below']} % a year"),
    (3, "Outside EMI", "Removed if the content is far from what EMI teaches.", "bar set by a calibration on OTH's own programmes"),
    (3, "Competition", "Counted by content similarity, whatever the programme is called. Not a cut.", "within 50, 100 and 150 km of Amberg"),
    (3, "North/south gap", "Does it exist elsewhere but not in Bavaria or the south?", "flags and a Bavarian quotient"),
    (5, "Job ads", "Keywords come from the type's own programme titles, never typed by hand.", f"counted in {len(RULES['labour']['regions'])} regions"),
    (6, "Minimum cohort", "Our own assumption; the supervisor gave none.", f"{c['min_cohort']['value']} first-years (central estimate)"),
    (7, "Overlap, lower end", "Share of credits covered by modules EMI already teaches.", f"at least {g['band']['min']:.0%}"),
    (7, "Overlap, upper end", "Share of credits that come from one existing OTH programme.", f"at most {g['one_programme']['max']:.0%}"),
    (7, "Existing facilities only", "New modules must be on topics a current lecturer already teaches. No new lab.", "checked by hand on two real curricula"),
    (8, "Decision", "Recommend the top candidate only if it is clearly ahead.", d["two_or_more"].replace("recommend the top one if ", "")),
]
show(pd.DataFrame(rows, columns=["Step", "Rule", "In plain words", "Number"]))

Step,Rule,In plain words,Number
2,Programme types,All full-time bachelor and master offers in Germany are grouped by their text. No topic list is written by hand.,a group is stable from an agreement of 0.75 across repeated runs
3,Already at OTH,A type is removed if an OTH programme of that level sits in its centre.,closer than half of the type's own programmes
3,Teacher training,Removed: a faculty cannot open a state teaching degree on its own.,from a share of 50%
3,Shrinking field,Removed if German first-years in Bavaria fall and the trend has predicted well in the past.,below -3.0 % a year
3,Outside EMI,Removed if the content is far from what EMI teaches.,bar set by a calibration on OTH's own programmes
3,Competition,"Counted by content similarity, whatever the programme is called. Not a cut.","within 50, 100 and 150 km of Amberg"
3,North/south gap,Does it exist elsewhere but not in Bavaria or the south?,flags and a Bavarian quotient
5,Job ads,"Keywords come from the type's own programme titles, never typed by hand.",counted in 3 regions
6,Minimum cohort,Our own assumption; the supervisor gave none.,20 first-years (central estimate)
7,"Overlap, lower end",Share of credits covered by modules EMI already teaches.,at least 40%


### The feasibility rule in one picture

```
overlap with ALL modules EMI already teaches
0 % ............ 40 % ...................................... 100 %
   too little            enough existing lectures to reuse
   (costs too many
    resources)

share that comes from ONE existing OTH programme
0 % ...................................... 80 % ............ 100 %
   a new programme                              a relabelled copy
                                                of that programme
```

A candidate passes if it is right of the 40 % line in the first bar and left of the 80 % line in the second. The recommendation is then taken only from candidates that existing staff and labs can carry.

In [4]:
# Where does every number in the rules come from? Each one carries a tag in the rules file.
TAGS = {"S": "the supervisor", "L": "literature or regulation", "C": "carried over from earlier pre-registered rules",
        "J": "our own judgement, tested with other values", "K": "written knowing a number of the first run (declared)",
        "N": "size or pass rule of a manual check"}
body = RULES_TEXT.split("definitions:", 1)[1]
counts = {t: len(re.findall(rf"\[{t}\]", body)) for t in TAGS}
show(pd.DataFrame([(f"[{t}]", TAGS[t], n) for t, n in counts.items()], columns=["Tag", "Origin of the number", "Times used"]))

Tag,Origin of the number,Times used
[S],the supervisor,8
[L],literature or regulation,3
[C],carried over from earlier pre-registered rules,39
[J],"our own judgement, tested with other values",16
[K],written knowing a number of the first run (declared),8
[N],size or pass rule of a manual check,8


In [5]:
# The stress tests: every judgement number is run with other values, to see whether the answer changes.
st = RULES["stress_tests"]
rows = [("after the candidate list is fixed", k.replace("_", " "), "; ".join(map(str, v)) if isinstance(v, list) else str(v)) for k, v in st["downstream_factorial"].items()]
rows += [("changes the candidate list", k.replace("_", " "), "; ".join(map(str, v)) if isinstance(v, list) else str(v)) for k, v in st["list_changing_one_at_a_time"].items()]
print(len(rows), "stress settings")
show(pd.DataFrame(rows, columns=["Kind", "What is varied", "Values"]))

31 stress settings


Kind,What is varied,Values
after the candidate list is fixed,min cohort,20; 25
after the candidate list is fixed,gross band,"[0.33, 3.0]; [0.5, 2.0]"
after the candidate list is fixed,pool trend,unprojected; projected; foreign added flat
after the candidate list is fixed,master transition,0.5; 0.3; 0.7
after the candidate list is fixed,dirichlet concentration,10; 50; 200
after the candidate list is fixed,overlap band,"[0.4, 0.8]; [0.5, 0.8]; [0.3, 0.9]"
after the candidate list is fixed,overlap reading,lower end on all EMI modules and upper end on one programme; both ends on all EMI modules
after the candidate list is fixed,one programme max,0.8; 0.5; none
after the candidate list is fixed,existing resources N cap,none; 20 bachelor and 10 master; 10 bachelor and 5 master
after the candidate list is fixed,references,both inside; primary only


### How the rules were checked

An independent reviewer, who had not seen our reasoning or any earlier result, attacked the rules three times.

| Round | Blocking problems found | Other major points | What happened |
|---|---|---|---|
| 1 | 11 | 10 | rules rewritten |
| 2 | 9 | 5 | rules reworded |
| 3 | 3 | 0 | reworded as proposed; not reviewed a fourth time |

| # | Criterion (written before the rules) | Target | Result | Pass |
|---|---|---|---|---|
| C1 | Every problem of the first run has a rule that prevents it | 14 of 14 | 14 of 14 | yes |
| C2 | Every point of the supervisor is in the rules, faithfully | 9 of 9 | 9 of 9 | yes |
| C3 | Every own-judgement number has a stress value | all | a listed group has none (mostly sizes of manual checks) | **no** |
| C4 | Every rule names its data source or a fallback | none missing | none missing | yes |
| C5 | No result produced or read in this step | none | only rule and document files changed | yes |
| C6 | No rule that two people could apply differently | 0 blockers | 0 after round 3 (last 3 fixes not re-reviewed) | yes |
| C7 | What we already knew is declared | complete | 12 rules marked as written knowing a first-run number | yes |
| C8 | No lecturer name in the repository | 0 | 0 | yes |
| C9 | All tests pass | all | 116 pass, 1 older test fails (not from this step) | **no** |

The two failed criteria were reported and accepted as exceptions; they are listed in `docs/funnel_preregistration.md`.

### What we achieved in Step 0

We now have one fixed rulebook for the whole analysis, written before any data is touched. It holds the supervisor's answers, a fix for every mistake of the first run, and a measure for his tip "popular elsewhere, missing in the south". Because the rules came first, the result cannot be steered towards a favourite programme.

**Strongest objection.** Some numbers are still our own judgement (for example the minimum of 20 students). **Answer.** Each of them is marked as such and is run with other values in Step 8, so we will see whether the recommendation depends on it.

## Next: Step 1, data inventory

- List every data source with its date, size and gaps: the study catalogue, Destatis student numbers, school leavers, job ads, EMI module handbooks, the timetables.
- Download what is missing and public: first-year students by subject for the other federal states (needed for the north/south check).
- Read the timetables into a table with neutral labels instead of lecturer names.
- No programme is looked at yet.